# DocumentLoaders

In [9]:
# Importar Bibliotecas Necessárias
import os
from dotenv import load_dotenv
import requests
from langchain_community.llms import Ollama
from langchain_ollama import OllamaLLM  , ChatOllama 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from IPython.display import Markdown, display

# Carregar variáveis de ambiente
load_dotenv()

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [10]:
# Verificar conexão com Ollama
OLLAMA_BASE_URL = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    if response.status_code == 200:
        modelos = response.json()
        print("✅ Conexão com Ollama estabelecida!")
        print(f"\n📦 Modelos disponíveis:")
        for modelo in modelos.get('models', []):
            print(f"  - {modelo['name']}")
    else:
        print("❌ Erro ao conectar com Ollama")
except requests.exceptions.ConnectionError:
    print("❌ Não foi possível conectar ao Ollama. Certifique-se que está rodando em Docker")

✅ Conexão com Ollama estabelecida!

📦 Modelos disponíveis:
  - nomic-embed-text:v1.5
  - qwen3:4b
  - olmo-3:7b
  - llama3.2:1b
  - llama3.1:8b
  - llama3.2:latest


### Tipos de Document Loaders no Langchain

![Document loaders](arquivos/loaders.png)

In [ ]:
Retrival Augmented Generation (RAG) é uma técnica que combina a geração de texto com a recuperação de informações relevantes. Em vez de depender apenas do conhecimento prévio do modelo, o RAG permite que o modelo acesse documentos ou dados externos para fornecer respostas mais precisas e informadas.

## Carregando PDFs

In [11]:
# from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader


caminiho2 = 'exemplos/arquivos/Explorando a API da OpenAI.pdf'
caminho = 'arquivos/Explorando o Universo das IAs com Hugging Face.pdf'
loader = PyPDFLoader(caminho)
documentos = loader.load()



In [ ]:
len(documentos)

In [13]:
print(documentos[17].page_content)

Explorando o Universo das IAs com Hugging Face
modelo = pipeline( 'fill-mask')
predicoes = modelo.predict( 'The capital of <mask> is Brasilia. ')
for predicao in predicoes:
resposta = predicao[ 'token_str']
score = predicao[ 'score']
frase = predicao[ 'sequence']
score_ajustado = score * 100
print(f'Predição "{resposta}" com score {score_ajustado:.2f}% -> "{frase}" ')
A saída deve ser algo como:
Predição " Brazil" com score 89.85% -> "The capital of Brazil is Brasilia."
Predição " Portugal" com score 1.26% -> "The capital of Portugal is Brasilia."
Predição " Europe" com score 1.10% -> "The capital of Europe is Brasilia."
Predição " crime" com score 0.60% -> "The capital of crime is Brasilia."
Predição " Angola" com score 0.47% -> "The capital of Angola is Brasilia."
Asimov Academy 17


In [14]:
documentos[3].metadata

{'producer': 'pdfTeX-1.40.25',
 'creator': 'LaTeX via pandoc with the Eisvogel template',
 'creationdate': '2024-03-20T18:50:05-03:00',
 'author': 'Asimov Academy',
 'title': 'Explorando o Universo das IAs com Hugging Face',
 'subject': '',
 'keywords': '',
 'moddate': '2024-03-20T18:50:05-03:00',
 'trapped': '/False',
 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023/Debian) kpathsea version 6.3.5',
 'source': 'arquivos/Explorando o Universo das IAs com Hugging Face.pdf',
 'total_pages': 89,
 'page': 3,
 'page_label': '3'}

### Fazendo perguntas para o arquivo

In [18]:
# from langchain.chains.question_answering import load_qa_chain
# from langchain_openai.chat_models import ChatOpenAI

from langchain_ollama import ChatOllama , OllamaEmbeddings
from langchain_chroma import Chroma

from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.runnables import RunnablePassthrough



chat = ChatOllama( model="llama3.2:1b" )

embedding = OllamaEmbeddings(model="nomic-embed-text:v1.5")

# vectorstore = Chroma(embedding_function=embedding)

# vectorstore = InMemoryVectorStore(documentos, embedding)

# crioando a vectorstore a partir dos documentos
vectorstore = Chroma.from_documents(
    documents=documentos[5:18], 
    embedding=embedding, 
    persist_directory='.db'
    )


retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

prompts = ChatPromptTemplate.from_messages([
    ("system", "Você é um assistente de pesquisa. Use o contexto abaixo para responder a pergunta."),
    ("human", "Contexto:\n{context}\n\nPergunta:\n{question}")
])

# chain = load_qa_chain(llm=chat, chain_type='stuff', verbose=True)

rag_chain = ( {"context": retriever,"question": RunnablePassthrough() } |
    prompts |
    chat
)
# pergunta = "Quais são as principais vantagens de usar a API da OpenAI?"

In [19]:
# Executar
pergunta = "Este curso não é exaustivo afinal, com mais de quantos modelos há no hugging face ?"

response = rag_chain.invoke(pergunta)

print(response.content)

A resposta para a pergunta é que há mais de 400.000 modelos no Hugging Face.


In [8]:
pergunta = 'Quais assuntos são tratados no documento?'

chain.stream(input_documents=documentos[:10], question=pergunta)

NameError: name 'chain' is not defined

## Carregando csv

In [7]:
from langchain_community.document_loaders.csv_loader import CSVLoader

caminho = 'arquivos/Top 1000 IMDB movies.csv'
loader = CSVLoader(caminho)
documentos = loader.load()

In [8]:
len(documentos)

1000

In [9]:
print(documentos[2].page_content)

: 2
Movie Name: The Dark Knight
Year of Release: (2008)
Watch Time: 152 min
Movie Rating: 9.0
Meatscore of movie: 84
Votes: 34,709
Gross: $534.86M
Description: When the menace known as the Joker wreaks havoc and chaos on the people of Gotham, Batman must accept one of the greatest psychological and physical tests of his ability to fight injustice.


In [10]:
documentos[2].metadata

{'source': 'arquivos/Top 1000 IMDB movies.csv', 'row': 2}

In [11]:
from langchain.chains.question_answering import load_qa_chain
from langchain_openai.chat_models import ChatOpenAI

chat = ChatOpenAI(model='gpt-3.5-turbo-0125')

chain = load_qa_chain(llm=chat, chain_type='stuff', verbose=True)

In [12]:
pergunta = 'Qual é o filme com maior metascore?'
chain.run(input_documents=documentos[:10], question=pergunta)



> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
System: Use the following pieces of context to answer the user's question. 
If you don't know the answer, just say that you don't know, don't try to make up an answer.
----------------
: 0
Movie Name: The Shawshank Redemption
Year of Release: (1994)
Watch Time: 142 min
Movie Rating: 9.3
Meatscore of movie: 81
Votes: 34,709
Gross: $28.34M
Description: Two imprisoned men bond over a number of years, finding solace and eventual redemption through acts of common decency.

: 1
Movie Name: The Godfather
Year of Release: (1972)
Watch Time: 175 min
Movie Rating: 9.2
Meatscore of movie: 100
Votes: 34,709
Gross: $134.97M
Description: The aging patriarch of an organized crime dynasty in postwar New York City transfers control of his clandestine empire to his reluctant youngest son.

: 2
Movie Name: The Dark Knight
Year of Release: (2008)
Watch Time: 152 min
Movie Rating: 9.0
Meatscore of movi

'O filme com o maior Meatscore é "The Godfather" com uma pontuação de 100.'

## Carregando da Internet

### Youtube

In [13]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.blob_loaders.youtube_audio import YoutubeAudioLoader
from langchain.document_loaders.parsers import OpenAIWhisperParser

In [14]:
url = 'https://www.youtube.com/watch?v=rOjusRRO1EI'
save_dir='docs/youtube/'
loader = GenericLoader(
    YoutubeAudioLoader([url], save_dir),
    OpenAIWhisperParser()
)
docs = loader.load()

[youtube] Extracting URL: https://www.youtube.com/watch?v=rOjusRRO1EI
[youtube] rOjusRRO1EI: Downloading webpage
[youtube] rOjusRRO1EI: Downloading tv player API JSON
[youtube] rOjusRRO1EI: Downloading ios player API JSON
[youtube] rOjusRRO1EI: Downloading m3u8 information
[info] rOjusRRO1EI: Downloading 1 format(s): 140
[download] docs/youtube//Como usar o GPT com seus próprios dados？.m4a has already been downloaded
[download] 100% of   25.62MiB
[ExtractAudio] Not converting audio docs/youtube//Como usar o GPT com seus próprios dados？.m4a; file is already in target format m4a
Transcribing part 1!


In [15]:
len(docs)

2

In [17]:
print(docs[0].page_content[:500])

Esse simples sistema aqui, essa telinha feia, com esse formulário aqui quase ridículo, é o tipo de projeto mais poderoso que vocês podem construir hoje em dia utilizando inteligência artificial. E o que isso aqui tem de tão especial? Esse sistema especificamente é capaz de responder perguntas sobre o funcionamento da Asimov porque ele foi treinado em uma base de conhecimentos que o nosso time está alimentando sobre como a empresa funciona. Então, nesse caso em específico, ele é capaz de responde


In [18]:
docs[1].metadata

{'source': 'docs/youtube/Como usar o GPT com seus próprios dados？.m4a',
 'chunk': 1}

### URLs

In [19]:
from langchain_community.document_loaders.web_base import WebBaseLoader

url = 'https://hub.asimov.academy/blog/listas-em-python/'
loader = WebBaseLoader(url)
documentos = loader.load()

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [20]:
len(documentos)

1

In [21]:
print(documentos[0].page_content[1000:2000])

ão em Python.
A lista em Python é uma das estruturas de dados fundamentais da linguagem Python. Além de possuir grande versatilidade, as listas são extremamente relevantes para iniciantes na programação, por incorporar uma variedade de conceitos básicos de Python como mutabilidade, indexação, iteração e slicing. Mas você já conhece as listas de Python a fundo?
Neste artigo, vamos nos aprofundar nas listas em Python e aprender a utilizá-las em seus códigos. Ao longo do texto, você aprenderá como criar e manipular uma lista em Python, quais os principais métodos de listas, e como elas se relacionam e com outros tipos de dados de Python, como strings, tuplas e vetores. Vamos lá!


      Curso Gratuito    

 

Seu primeiro projeto Python – curso grátis com certificado!
Vá do zero ao primeiro projeto em apenas 2 horas com o curso Python para Iniciantes.
Comece agora


O que é uma lista em Python?
Uma lista em Python é uma estrutura de dados que armazena uma sequência de valores. As listas e

## Carregando do Notion

In [25]:
from langchain_community.document_loaders.notion import NotionDirectoryLoader
caminho = 'arquivos/notion_db'
loader = NotionDirectoryLoader(caminho)
documentos = loader.load()

In [ ]:
len(documentos)

In [ ]:
print(documentos[0].page_content)

In [19]:
documentos[2].metadata

{'source': 'arquivos\\notion_db\\Wiki da Asimov 9d5b5c18267b4db2af0e216e8d418745\\Arrumar dados do Database 98cb8cde71cc489e8be9ba405bf19596.md'}